In [1]:
#Importing libraries for the dataset
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

In [2]:
#Recursive Feature Elimination defining funtions with multiple algorithms
def rfeFeature(indep_X,dep_Y,n):
    rfeList=[]
    log_model = LogisticRegression(solver='lbfgs', max_iter=100)
    RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    DT= DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
    svc_model = SVC(kernel = 'linear', random_state = 0)
    #NB = GaussianNB()
    rfemodellist=[log_model,svc_model,RF,DT] 
    for i in rfemodellist:
        print(i)
        log_rfe = RFE(estimator=i,n_features_to_select=n)
        log_fit = log_rfe.fit(indep_X, dep_Y)
        log_rfe_feature=log_fit.transform(indep_X)
        mask=log_fit.get_support()
        selected_features=indep_X.columns[mask].tolist()
        print(selected_features)
        rfeList.append(log_rfe_feature)

    return selected_features
    
    

In [3]:
#Applying standard scalar
def split_scalar(indep_X,dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    return X_train, X_test, y_train, y_test

In [4]:
def cm_prediction(classifier,X_test,y_test):
    y_pred = classifier.predict(X_test)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred)
        
    from sklearn.metrics import accuracy_score 
    from sklearn.metrics import classification_report        
    Accuracy=accuracy_score(y_test, y_pred )
    report=classification_report(y_test, y_pred)
    return  classifier,Accuracy,report,X_test,y_test,cm

In [5]:
def logistic(X_train,y_train,X_test,y_test):       
    from sklearn.linear_model import LogisticRegression
    classifier = LogisticRegression(random_state = 0)
    classifier.fit(X_train, y_train)
    classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
    return  classifier,Accuracy,report,X_test,y_test,cm      


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def logistic(X_train, y_train, X_test, y_test):
    classifier = LogisticRegression()
    classifier.fit(X_train, y_train)
    y_pred = classifier.predict(X_test)
    
    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    return classifier, Accuracy, report, X_test, y_test, cm


In [7]:
def svm_linear(X_train,y_train,X_test,y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel = 'linear', random_state = 0)
    classifier.fit(X_train, y_train)
    classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
    return  classifier,Accuracy,report,X_test,y_test,cm

In [8]:
def Navie(X_train,y_train,X_test,y_test):       
    from sklearn.naive_bayes import GaussianNB
    classifier = GaussianNB()
    classifier.fit(X_train, y_train)
    classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
    return  classifier,Accuracy,report,X_test,y_test,cm         


In [9]:
def Decision(X_train,y_train,X_test,y_test):
    
    # Fitting K-NN to the Training set
    from sklearn.tree import DecisionTreeClassifier
    classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
    classifier.fit(X_train, y_train)
    classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
    return  classifier,Accuracy,report,X_test,y_test,cm      


In [10]:
def random(X_train,y_train,X_test,y_test):
        
    # Fitting K-NN to the Training set
    from sklearn.ensemble import RandomForestClassifier
    classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
    classifier.fit(X_train, y_train)
    classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
    return  classifier,Accuracy,report,X_test,y_test,cm
    


In [11]:
def rfe_classification(acclog,accsvml,accdes,accrf): 
    
    rfedataframe=pd.DataFrame(index=['Logistic','SVC','Random','DecisionTree'],columns=['Logistic','SVMl','Decision','Random'])
    
    for number,idex in enumerate(rfedataframe.index):
        
        rfedataframe['Logistic'][idex]=acclog[number]       
        rfedataframe['SVMl'][idex]=accsvml[number]
        rfedataframe['Decision'][idex]=accdes[number]
        rfedataframe['Random'][idex]=accrf[number]
    return rfedataframe



In [12]:
#Reading the file
dataset=pd.read_csv("border_intrusion_dataset.csv")

In [13]:
#To view the dataset
dataset

,sensor_id,timestamp_hour,motion_intensity,vibration_level,thermal_signature,acoustic_level,object_speed_kmph,distance_m,altitude_m,water_pressure,route_mode,weather_condition,terrain_type,sensor_status,signal_noise_ratio,target_class,weight
0,1001,15,7.87,77.87,710.60,77.87,7.87,710.60,0.00,0.00,Ground,Clear,Mountain,Active,710.60,Human,70
1,1002,6,4.29,75.29,145.61,74.29,4.29,145.61,0.00,0.00,Ground,Clear,Forest,Active,145.61,Human,70
2,1003,21,7.11,79.11,302.38,77.11,7.11,302.38,0.00,0.00,Ground,Clear,Open Field,Active,302.38,Human,70
3,1004,1,4.94,75.94,589.96,74.94,4.94,589.96,0.00,0.00,Ground,Clear,Forest,Active,589.96,Human,70
4,1005,23,5.28,75.28,499.80,75.28,5.28,499.80,0.00,0.00,Ground,Clear,Mountain,Active,499.80,Human,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,1346,6,1.02,61.02,420.29,61.02,1.02,415.29,0.00,0.00,Ground,Clear,Mountain,Active,415.29,False Alarm,60
346,1347,6,1.32,62.32,133.90,61.32,1.32,128.90,0.00,3.05,Water Route,Storm,Forest,Active,130.90,False Alarm,60
347,1348,13,1.84,63.84,368.89,61.84,1.84,363.89,0.00,0.00,Ground,Rain,Open Field,Active,364.89,False Alarm,60
348,1349,14,0.96,60.96,277.58,60.96,0.96,272.58,23.15,0.00,Air Route,Clear,Mountain,Degraded,272.58,False Alarm,60


In [14]:
# To view Null values
dataset.isnull().sum()

sensor_id              0
timestamp_hour         0
motion_intensity       0
vibration_level        0
thermal_signature      0
acoustic_level         0
object_speed_kmph      0
distance_m             0
altitude_m             0
water_pressure         0
route_mode             0
weather_condition     10
terrain_type           0
sensor_status          0
signal_noise_ratio     0
target_class           0
weight                 0
dtype: int64

In [15]:
#Using for loop to fill the null value
for col in ['weather_condition']:
    dataset.loc[:, col] = dataset[col].fillna(dataset[col].mode()[0])

In [16]:
dataset

,sensor_id,timestamp_hour,motion_intensity,vibration_level,thermal_signature,acoustic_level,object_speed_kmph,distance_m,altitude_m,water_pressure,route_mode,weather_condition,terrain_type,sensor_status,signal_noise_ratio,target_class,weight
0,1001,15,7.87,77.87,710.60,77.87,7.87,710.60,0.00,0.00,Ground,Clear,Mountain,Active,710.60,Human,70
1,1002,6,4.29,75.29,145.61,74.29,4.29,145.61,0.00,0.00,Ground,Clear,Forest,Active,145.61,Human,70
2,1003,21,7.11,79.11,302.38,77.11,7.11,302.38,0.00,0.00,Ground,Clear,Open Field,Active,302.38,Human,70
3,1004,1,4.94,75.94,589.96,74.94,4.94,589.96,0.00,0.00,Ground,Clear,Forest,Active,589.96,Human,70
4,1005,23,5.28,75.28,499.80,75.28,5.28,499.80,0.00,0.00,Ground,Clear,Mountain,Active,499.80,Human,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,1346,6,1.02,61.02,420.29,61.02,1.02,415.29,0.00,0.00,Ground,Clear,Mountain,Active,415.29,False Alarm,60
346,1347,6,1.32,62.32,133.90,61.32,1.32,128.90,0.00,3.05,Water Route,Storm,Forest,Active,130.90,False Alarm,60
347,1348,13,1.84,63.84,368.89,61.84,1.84,363.89,0.00,0.00,Ground,Rain,Open Field,Active,364.89,False Alarm,60
348,1349,14,0.96,60.96,277.58,60.96,0.96,272.58,23.15,0.00,Air Route,Clear,Mountain,Degraded,272.58,False Alarm,60


In [17]:
dataset.isnull().sum()

sensor_id             0
timestamp_hour        0
motion_intensity      0
vibration_level       0
thermal_signature     0
acoustic_level        0
object_speed_kmph     0
distance_m            0
altitude_m            0
water_pressure        0
route_mode            0
weather_condition     0
terrain_type          0
sensor_status         0
signal_noise_ratio    0
target_class          0
weight                0
dtype: int64

In [18]:
dataset=pd.get_dummies(dataset,drop_first=True)
dataset

,sensor_id,timestamp_hour,motion_intensity,vibration_level,thermal_signature,acoustic_level,object_speed_kmph,distance_m,altitude_m,water_pressure,...,weather_condition_Storm,terrain_type_Mountain,terrain_type_Open Field,terrain_type_Riverbank,sensor_status_Degraded,target_class_Animal,target_class_False Alarm,target_class_Human,target_class_Vehicle,target_class_Water Intrusion
0,1001,15,7.87,77.87,710.60,77.87,7.87,710.60,0.00,0.00,...,False,True,False,False,False,False,False,True,False,False
1,1002,6,4.29,75.29,145.61,74.29,4.29,145.61,0.00,0.00,...,False,False,False,False,False,False,False,True,False,False
2,1003,21,7.11,79.11,302.38,77.11,7.11,302.38,0.00,0.00,...,False,False,True,False,False,False,False,True,False,False
3,1004,1,4.94,75.94,589.96,74.94,4.94,589.96,0.00,0.00,...,False,False,False,False,False,False,False,True,False,False
4,1005,23,5.28,75.28,499.80,75.28,5.28,499.80,0.00,0.00,...,False,True,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,1346,6,1.02,61.02,420.29,61.02,1.02,415.29,0.00,0.00,...,False,True,False,False,False,False,True,False,False,False
346,1347,6,1.32,62.32,133.90,61.32,1.32,128.90,0.00,3.05,...,True,False,False,False,False,False,True,False,False,False
347,1348,13,1.84,63.84,368.89,61.84,1.84,363.89,0.00,0.00,...,False,False,True,False,False,False,True,False,False,False
348,1349,14,0.96,60.96,277.58,60.96,0.96,272.58,23.15,0.00,...,False,True,False,False,True,False,True,False,False,False


In [19]:
dataset.isnull().sum()

sensor_id                       0
timestamp_hour                  0
motion_intensity                0
vibration_level                 0
thermal_signature               0
acoustic_level                  0
object_speed_kmph               0
distance_m                      0
altitude_m                      0
water_pressure                  0
signal_noise_ratio              0
weight                          0
route_mode_Ground               0
route_mode_Vehicle Route        0
route_mode_Water Route          0
weather_condition_Fog           0
weather_condition_Rain          0
weather_condition_Storm         0
terrain_type_Mountain           0
terrain_type_Open Field         0
terrain_type_Riverbank          0
sensor_status_Degraded          0
target_class_Animal             0
target_class_False Alarm        0
target_class_Human              0
target_class_Vehicle            0
target_class_Water Intrusion    0
dtype: int64

In [20]:
indep_X=dataset.drop(['target_class_Animal','target_class_False Alarm','target_class_Human','target_class_Vehicle','target_class_Water Intrusion'],axis=1)

In [21]:
dep_Y=dataset[['target_class_Animal','target_class_False Alarm','target_class_Human','target_class_Vehicle','target_class_Water Intrusion']].idxmax(axis=1)

In [22]:
rfeList=rfeFeature(indep_X,dep_Y,3)

LogisticRegression()


C:\Anaconda\envs\aiml\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda\envs\aiml\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to

['motion_intensity', 'object_speed_kmph', 'water_pressure']
SVC(kernel='linear', random_state=0)
['object_speed_kmph', 'water_pressure', 'weight']
RandomForestClassifier(criterion='entropy', n_estimators=10, random_state=0)
['sensor_id', 'object_speed_kmph', 'weight']
DecisionTreeClassifier(max_features='sqrt', random_state=0)
['signal_noise_ratio', 'weight', 'route_mode_Water Route']


In [23]:
acclog=[]
accsvml=[]
accdes=[]
accrf=[]

for i in rfeList:   
    X_train, X_test, y_train, y_test=split_scalar(i,dep_Y)   
    classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test,y_test)
    acclog.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test,y_test)  
    accsvml.append(Accuracy)

    
    classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test,y_test)  
    accdes.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test,y_test)  
    accrf.append(Accuracy)
    
result=rfe_classification(acclog,accsvml,accdes,accrf)

ValueError: Found input variables with inconsistent numbers of samples: [18, 350]

In [ ]:
result

In [ ]:
selector=rfeFeature(indep_X,dep_Y,3)
mask=selector[0].get_support()
selected_features = indep_X.columns[mask].tolist()
print(selected_features)